In [1]:
import pandas as pd
import os
from dotenv import load_dotenv, find_dotenv
from pinecone import Pinecone, ServerlessSpec
import pinecone
import time
from sentence_transformers import SentenceTransformer

d:\projects\VectorDB Semantic Search\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(find_dotenv(), override = True)

True

In [4]:
files = pd.read_csv(r"D:\projects\VectorDB Semantic Search\course-section-descriptions.csv", encoding = "ANSI")

In [5]:
files["unique_id"] = files["course_id"].astype(str) + '-' + files["section_id"].astype(str)

In [6]:
files["metadata"] = files.apply(lambda row: {
    "course_name": row["course_name"],
    "section_name": row["section_name"],
    "section_description": row["section_description"],
}, axis = 1)

In [7]:
def create_embeddings(row):
    combined_text = f'''{row["course_name"]} {row["course_technology"]}
    {row["course_description"]} {row["section_name"]}{row["section_description"]}'''
    return model.encode(combined_text, show_progress_bar = False)

In [8]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1349.94it/s]


In [ ]:
files["embedding"] = files.apply(create_embeddings, axis = 1)

Upserting Data to Pinecone

In [9]:
pc = Pinecone(api_key = os.environ.get("PINECONE_API_KEY"), environment = os.environ.get("PINECONE_ENV"))

In [10]:
index_name = "my-index"
dimension = 384
metric = "cosine"

In [11]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} successfully deleted.")
else:
    print(f"{index_name} not in index list.")

my-index successfully deleted.


In [12]:
pc.create_index(
    name = index_name,
    dimension = dimension,
    metric = metric,
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
    )
)

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-fh1kny3.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [13]:
index = pc.Index(index_name)